In [1]:
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/limpieza/cleaned_data.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volume_binance,trades_binance,volumen_binance,SMA_50_binance,SMA_200_binance,...,volatilidad_200_dxy,momentum_50_dxy,momentum_200_dxy,var_diaria_dxy,rango_dia_dxy,RSI_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-01-30,23743.37,23800.51,22500.00,22826.15,302405.90121,7790224.0,6.902762e+09,18840.4428,19681.15235,...,3.344820,-2.743332,-6.260002,0.114196,0.649994,48.159051,0.887539,1,15.651634,1
1,2023-01-31,22827.38,23320.00,22714.77,23125.13,264649.34909,6798411.0,6.120051e+09,18958.7488,19692.62780,...,3.365946,-3.029999,-5.959999,-0.175988,0.599998,42.076453,0.651979,1,14.475111,1
2,2023-02-01,23125.13,23812.66,22760.23,23732.66,310790.42271,7863213.0,7.375883e+09,19077.9080,19705.31310,...,3.394349,-2.760002,-6.609998,-0.861897,1.150002,28.731297,-0.237540,-1,12.944154,1
3,2023-02-02,23731.41,24255.00,23363.27,23488.94,364177.20751,9102300.0,8.554137e+09,19191.6238,19718.76700,...,3.417732,-2.019997,-5.850001,0.523611,1.090004,44.673560,-1.569186,-1,11.529444,1
4,2023-02-03,23489.33,23715.70,23204.62,23431.90,332571.02904,8007846.0,7.792771e+09,19313.1350,19723.76360,...,3.432184,-1.639999,-4.450005,1.149875,1.459999,61.290298,-2.890628,-1,10.208064,1


# Agrupar variables

In [3]:
sufijos = [
    "_binance",
    "_fg",
    "_gtrends",
    "_hashrate",
    "_mDbinance",
    "_sp500",
    "_nasdaq",
    "_dxy",
]

# Crear un diccionario para almacenar las columnas separadas
columnas_por_sufijo = {sufijo: [] for sufijo in sufijos}

# Recorrer las columnas y clasificarlas
for col in df.columns:
    for sufijo in sufijos:
        if col.endswith(sufijo):
            columnas_por_sufijo[sufijo].append(col)
            break

# Ahora tienes las columnas organizadas por sufijo
suma = 5 # timestamp, 4 columnas respuesta
for sufijo, columnas in columnas_por_sufijo.items():
    print(f"\nSufijo {sufijo} ({len(columnas)}):")
    print(columnas)
    suma += len(columnas)

assert suma == df.shape[1], f"Suma: {suma}, columnas:{df.shape[1]}"


Sufijo _binance (18):
['open_binance', 'high_binance', 'low_binance', 'close_binance', 'volume_binance', 'trades_binance', 'volumen_binance', 'SMA_50_binance', 'SMA_200_binance', 'EMA_50_binance', 'EMA_200_binance', 'var_diaria_binance', 'volatilidad_50_binance', 'volatilidad_200_binance', 'rango_dia_binance', 'momentum_50_binance', 'momentum_200_binance', 'RSI_binance']

Sufijo _fg (1):
['fear_greed_index_fg']

Sufijo _gtrends (1):
['bitcoin_gtrends']

Sufijo _hashrate (5):
['hashrate_hashrate', 'SMA_50_hashrate', 'SMA_200_hashrate', 'EMA_50_hashrate', 'EMA_200_hashrate']

Sufijo _mDbinance (17):
['open_mDbinance', 'high_mDbinance', 'low_mDbinance', 'close_mDbinance', 'volume_mDbinance', 'trades_mDbinance', 'var_diaria_mDbinance', 'volatilidad_50_mDbinance', 'volatilidad_200_mDbinance', 'rango_dia_mDbinance', 'momentum_50_mDbinance', 'momentum_200_mDbinance', 'SMA_50_mDbinance', 'SMA_200_mDbinance', 'EMA_50_mDbinance', 'EMA_200_mDbinance', 'RSI_mDbinance']

Sufijo _sp500 (16):
['clos

# Eliminación de variables redundantes o con información duplicada


## 1. Binance – precios OHLC
**Eliminar:**
- open_binance
- high_binance
- low_binance
- volumen_binance: se calcula close_binance * volume_binance

**Dejar:**
- close_binance
- volume_binance
- trades_binance
- rango_dia_binance: diferencia de high y low
- var_diaria_binance: cambio porcentual entre un valor y el valor inmediatamente anterior (en porcentaje)

In [4]:
df.drop(columns=['open_binance', 'high_binance', 'low_binance', 'volumen_binance'], inplace=True)

## 2. Binance - Medias móviles
Mantener una combinación de corto y largo plazo, evitando duplicar SMA y EMA para ambos.

**Eliminar:**
- SMA_50_binance
- EMA_200_binance

**Dejar:**
- EMA_50_binance
- SMA_200_binance

In [5]:
df.drop(columns=['SMA_50_binance', 'EMA_200_binance'], inplace=True)

## 3. Binance - Otras varibles
Conservaremos las variables de volatilidad, momentum y RSI

## 4. Indicadores
Conservaremos los indicadores: `fear_greed_index_fg` y `bitcoin_gtrends`

## 5. Hashrate
Conservaremos la variable `hashrate_hashrate` y mantendremos una combinación de corto y largo plazo para las medias móviles como en Binance

**Eliminar:**
- SMA_50_hashrate
- EMA_200_hashrate

**Dejar:**
- EMA_50_hashrate
- SMA_200_hashrate

In [6]:
df.drop(columns=['SMA_50_hashrate', 'EMA_200_hashrate'], inplace=True)

## 6. Mercado Derivados Binance, S&P 500, Nasdaq, DXY
Al compartir las mismas variables que Binance, repetiremos el proceso:

**Eliminar:**
- open
- high
- low
- SMA_50
- EMA_200

**Dejar:**
- close
- volume
- trades
- var_diaria
- rango dia
- volatilidad (50/200)
- momentum (50/200)
- SMA_200
- EMA_50

In [7]:
# mDbinance
df.drop(columns=[
    'open_mDbinance',
    'high_mDbinance',
    'low_mDbinance',
    'SMA_50_mDbinance',
    'EMA_200_mDbinance'
        ],
        inplace=True)

# S&P 500, Nasdaq, DXY
sufijos = ['_sp500', '_nasdaq', '_dxy']
columnas = ['open', 'high', 'low', 'SMA_50', 'EMA_200']
columnas_a_eliminar = [col + suf for col in columnas for suf in sufijos]

df.drop(columns=columnas_a_eliminar, inplace=True)

## 7. Varibles respuesta categóricas
Para mostrar la creación de la variable respuesta gráficamente se creó una variable categórica auxiliar que indicaba la tendencia alcista o bajista del bitcoin. Como no se va a realizar problemas de clasificación se eliminarán estas variables

In [8]:
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

## 8. Conjunto de datos para 7 días
No se eliminará ninguna otra variable salvo la variable respuesta para 30 días

In [9]:
df_7d = df.drop(columns=['pct_cambio_30d'])

output_path = r"../../data/limpieza/dataset_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

## 9. Conjunto de datos para 30 días
No se eliminará ninguna otra variable salvo la variable respuesta para 7 días

In [10]:
df_30d = df.drop(columns=['pct_cambio_7d'])

output_path = r"../../data/limpieza/dataset_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

In [11]:
df.head()

,timestamp,close_binance,volume_binance,trades_binance,SMA_200_binance,EMA_50_binance,var_diaria_binance,volatilidad_50_binance,volatilidad_200_binance,rango_dia_binance,...,EMA_50_dxy,volatilidad_50_dxy,volatilidad_200_dxy,momentum_50_dxy,momentum_200_dxy,var_diaria_dxy,rango_dia_dxy,RSI_dxy,pct_cambio_7d,pct_cambio_30d
0,2023-01-30,22826.15,302405.90121,7790224.0,19681.15235,19839.933272,-3.858725,2606.068847,2406.761982,1300.51,...,103.437153,1.076617,3.344820,-2.743332,-6.260002,0.114196,0.649994,48.159051,0.887539,15.651634
1,2023-01-31,23125.13,264649.34909,6798411.0,19692.62780,19968.764516,1.309814,2664.153492,2417.713990,605.23,...,103.384715,1.060349,3.365946,-3.029999,-5.959999,-0.175988,0.599998,42.076453,0.651979,14.475111
2,2023-02-01,23732.66,310790.42271,7863213.0,19705.31310,20116.368261,2.627142,2742.210819,2432.251472,1052.43,...,103.299825,1.095670,3.394349,-2.760002,-6.609998,-0.861897,1.150002,28.731297,-0.237540,12.944154
3,2023-02-02,23488.94,364177.20751,9102300.0,19718.76700,20248.625976,-1.026939,2805.431757,2445.731503,891.73,...,103.239047,1.113146,3.417732,-2.019997,-5.850001,0.523611,1.090004,44.673560,-1.569186,11.529444
4,2023-02-03,23431.90,332571.02904,8007846.0,19723.76360,20373.460252,-0.242838,2855.447164,2452.315588,511.08,...,103.226536,1.096533,3.432184,-1.639999,-4.450005,1.149875,1.459999,61.290298,-2.890628,10.208064
